In [1]:
from dotenv import load_dotenv
import getpass
import os

if not os.getenv("HUGGINGFACEHUB_API_TOKEN"):
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = getpass.getpass("Enter your token:")

load_dotenv()

True

### Load Data, Chat Model and Embedding Model

In [ ]:
import importlib
import eval_utils

importlib.reload(eval_utils)

/home/a.zhang@PTW.Maschinenbau.TU-Darmstadt.de/project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<module 'eval_utils' from '/home/a.zhang@PTW.Maschinenbau.TU-Darmstadt.de/project/RAGAS/eval_utils.py'>

In [3]:
from eval_utils import load_llm, load_embeddings, load_docs

# load chat model
chat_model = load_llm(model_path="./hf_models/Llama-3.1-8B-Instruct")

# load embedding model
embedding = load_embeddings(model_id="sentence-transformers/all-MiniLM-L6-v2",
                            cache_dir="hf_models/embeddings/all-MiniLM-L6-v2")

# load docs
docs = load_docs("data/mk4s_manuel")

Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.39it/s]
Device set to use cuda:0
/home/a.zhang@PTW.Maschinenbau.TU-Darmstadt.de/project/RAGAS/eval_utils.py:55: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(
100%|██████████| 8/8 [00:00<00:00, 9554.22it/s]


### Use Recursive Character Chunking to split

In [4]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=256, 
                                               chunk_overlap=32,
                                               separators=["\n## ","\n### ","\n#### ", "\n\n", ".", "\n", " "],
                                               is_separator_regex=False,
                                               keep_separator=False,)
chunks = text_splitter.split_documents(docs)

### Set Hybrid Retriver(bm25:0.4 + mmr:0.6)

In [5]:
import time
from typing import List, Set
import pandas as pd
from langchain_community.vectorstores import FAISS
from langchain_core.retrievers import BaseRetriever
from langchain.retrievers.ensemble import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_core.prompts import PromptTemplate
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_core.runnables import RunnableLambda
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_core.documents import Document


vector_storage = FAISS.from_documents(chunks, embedding)

bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 10

faiss_retriever = vector_storage.as_retriever(
                                search_type="mmr",
                                search_kwargs={"k": 10, "lambda_mult": 0.7}
                                )
class HybridRetriever(BaseRetriever):
    
    base_retriever: BaseRetriever

    def _get_relevant_documents(
        self,
        query: str,
        *,
        run_manager: CallbackManagerForRetrieverRun,
    ) -> List[Document]:
        
        docs = self.base_retriever.get_relevant_documents(query)

        # deduplication
        seen: Set[str] = set()
        unique_docs: List[Document] = []
        for d in docs:
            if d.page_content not in seen:
                seen.add(d.page_content)
                unique_docs.append(d)

        return unique_docs
    
ensemble_retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, faiss_retriever], weights=[0.4, 0.6]
    )

hybrid_retriever = HybridRetriever(base_retriever=ensemble_retriever)

### Set multi-query retriever - generate 4 (include original query) queries

In [6]:
# mq_prompt = PromptTemplate(
#     input_variables=["question"],
#     template=(
#         """ You are a helpful research assistant. 
#         Given the user's question below, generate **exactly three** alternative search queries. 
# 
#         Each query should capture a different aspect or phrasing of the same information need.
#         Avoid redundancy, keep them short and clear.
#         User question: {question}
#         Provide the 3 reformulated search queries, each on a new line:
#         """
#     ),
# )
from langchain.prompts import PromptTemplate, FewShotPromptTemplate

#  few-shot template
example_prompt = PromptTemplate(
    input_variables=["question", "q1", "q2", "q3"],
    template=(
        "User question: {question}\n"
        "Provide the 3 reformulated search queries, each on a new line:\n"
        "{q1}\n"
        "{q2}\n"
        "{q3}\n"
    ),
)

# 2. exapmles
examples = [
    {
        "question": "What is extruder blob in 3D printing?",
        "q1": "What causes extruder blob formation in 3D printing?",
        "q2": "How do extruder blobs develop on FDM printers?",
        "q3": "What is an extruder blob and why does it appear?",
    },
    {
        "question": "What happen with the bigger blobs on the printer?",
        "q1": "What issues can large extruder blobs create on a printer?",
        "q2": "How do big filament blobs impact printing performance?",
        "q3": "What occurs when a large blob forms on the hotend?",
    },
    {
        "question": "What should I check if I encounter a Preheat error related to the thermistor leads before starting a print?",
        "q1": "How can I troubleshoot a preheat error caused by the thermistor leads before printing?",
        "q2": "Which parts should I inspect when a preheat error indicates an issue with the thermistor wiring?",
        "q3": "What checks are required on the thermistor leads if the printer reports a preheat error?",
    },
    {
        "question": "What are the recommended steps for checking belt tension and ensuring proper motor pulley alignment on both the Original Prusa MK-series and Original Prusa XL printers?",
        "q1": "How do I verify belt tension and motor pulley alignment on Original Prusa MK-series and XL printers?",
        "q2": "What is the procedure for checking X/Y belt tension and adjusting motor pulleys on Prusa MK and XL models?",
        "q3": "Which steps should I follow to inspect belts and pulley alignment on Prusa MK-series and XL machines?",
    },
]

# 3. generate final MultiQuery Prompt
mq_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix=(
        "You are a helpful research assistant.\n"
        "Your task is to rewrite the user's question into EXACTLY THREE alternative search queries.\n"
        "The queries must be short, clear, and capture different aspects of the same information need.\n"
        "IMPORTANT:\n"
        "- DO NOT add explanations.\n"
        "- DO NOT add numbering.\n"
        "- DO NOT add commentary.\n"
        "- Output MUST contain ONLY three lines.\n"
    ),
    suffix=(
        "User question: {question}\n\n"
        "Three reformulated search queries:\n"
    ),
    input_variables=["question"],
)

# mq_prompt.format(question="Why is my 3D print stringing?")


# Create MultiQueryRetriever，prompt and LLM
mq_retriever = MultiQueryRetriever.from_llm(
    retriever=hybrid_retriever,
    llm=chat_model,
    prompt=mq_prompt,
    include_original=True,
)

In [7]:
# test output queries
question = 'how do i kno if i succed or not with the extruder'
generated_queries = mq_retriever.llm_chain.invoke({"question": question})
print("=== Generated Query List ===")
for i, q in enumerate(generated_queries, 1):
    print(f"{i}. {q}")

print("\n=== Running retrieval ===")
docs = mq_retriever.get_relevant_documents(question)

for i, doc in enumerate(docs[:3], 1):
    print(f"Doc {i}: {doc.metadata if hasattr(doc, 'metadata') else ''}")
    print(doc.page_content[:200], "...\n")

=== Generated Query List ===
1. How to determine successful extruder calibration
2. Extruder calibration verification steps for 3D printing
3. Checking extruder accuracy after calibration adjustments

=== Running retrieval ===


/tmp/ipykernel_1177487/3266641430.py:9: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = mq_retriever.get_relevant_documents(question)


Doc 1: {'source': 'data/mk4s_manuel/stringing_and_oozing.md'}
Make sure there are **no obstructions** in the path of the extruder or heatbed and their bearings. For example, there might be a piece of filament stuck around the belt (usually around the Y-axis pull ...

Doc 2: {'source': 'data/mk4s_manuel/stringing_and_oozing.md'}
Make sure the extruder and the heatbed can move freely ...

Doc 3: {'source': 'data/mk4s_manuel/Layer_shifting.md'}
** It is usually associated with an abnormal movement of the X-axis and/or the Y-axis, leading the extruder head to be misaligned mid-printing. ...



### Deduplication + Reranker : create contextual(2nd-retrieval) pipeline Return top-15

In [8]:
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_community.document_transformers import EmbeddingsRedundantFilter
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import DocumentCompressorPipeline

reranker_model = HuggingFaceCrossEncoder(model_name="hf_models/rerank/bge-reranker-large",
                                         model_kwargs={"device": "cuda:0"},)
redundant_filter = EmbeddingsRedundantFilter(embeddings=embedding, similarity_threshold=0.9)


reranker  = CrossEncoderReranker(model=reranker_model,top_n=15)

pipeline_compressor = DocumentCompressorPipeline(
        transformers=[redundant_filter,reranker]
    )

compression_retriever = ContextualCompressionRetriever( 
        base_compressor=pipeline_compressor,
        base_retriever=mq_retriever
    )

### Create Few-Shots RAG Chain

In [ ]:
# prompt
from langchain.prompts import PromptTemplate, FewShotPromptTemplate

examples = [
    {
        "question": "What is extruder blob in 3D printing?",
        "answer": (
            """Extruder blob is a mass of plastic accumulated around the hotend, and it is one of the most scary-looking printing problems you might face with your 3D printer."""
        ),
    },
    {
        "question": "What happen with the bigger blobs on the printer?",
        "answer": (
            """As for the bigger blobs, they will not fall off by themselves like the smaller ones."""
        ),
    },
    {
        "question": "What steps should I take to check the belt tension on my Original Prusa MK3 and how can I address an extruder blob issue?",
        "answer": (
            """To prevent extruder blobs while using the Original Prusa MK3, you should follow these steps: First, always clean the print surface before starting the print to ensure proper adhesion. Second, adjust the first layer height at the beginning of the print to make sure that the first layer sticks properly to the entire print surface. Additionally, monitor the print periodically; if you notice any potential issues, pause the print to address them. As a last resort, you can stop the print, adjust the settings, and start again. For optimal printer performance, ensure that your printer is running in normal mode rather than stealth mode, as stealth mode disables the crash detection feature. Also, check that the extruder and heatbed can move freely, verify that the smooth rods are clean and lubricated, and ensure that the X/Y axis motors and pulleys are properly tightened and aligned."""
        ),
    },
    {
        "question": "What are the recommended steps for checking belt tension and ensuring proper motor pulley alignment on both the Original Prusa MK-series and Original Prusa XL printers?",
        "answer": (
            """For the Original Prusa MK-series, to check the belt tension, you can use the belt tuner or check the Belt Status numbers via the LCD Menu under Support. The optimal tension values are approximately 250 on the X-axis and 275 on the Y-axis, with a tolerance of ±15. If the value is below 240, the belt needs to be loosened, and if it's above 290, it needs to be tightened. Additionally, ensure that the X and Y motors are securely tightened in their mounts and that the pulleys are aligned and can move freely. For the Original Prusa XL, check the belt tension using the guidelines in the dedicated article and inspect the belt clamps for any visible damage. If a motor pulley has loosened, it can cause misalignment, so ensure that the pulleys are positioned correctly: the left pulley should be 2.5mm higher than the flat part of the motor shaft, while the right pulley should be flush with the top edge of the motor shaft."""

        ),
    },
]

example_prompt = PromptTemplate(
    input_variables=["question", "answer"],
    template="Question: {question}\n Answer: {answer}\n",
)

prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix=(
        """
        You are a helpful assistant specializing in 3D printing operations in shopfloor environment.
        Detect whether the question is a SINGLE-HOP question or a MULTI-HOP question.(Do not add Detection in answer)
            - If it is SINGLE-HOP:answer in the style of Examples 1 and 2.
            - If it is MULTI-HOP: answer in the style of Examples 3 and 4.
        Answers should be concise, oriented toward clear next actions and basic maintenance checks.
        If you do not know the answer, respond: "I don't know."

        Below are examples of the expected behavior:
        """
    ),
    suffix=(
        """ 
        Question: {query}
        Context: {context}
        Your answer must be strictly based on the retrieved context.
        Answer:
        """
    ),
    input_variables=["question"],
)


In [ ]:
import time
import pandas as pd
from langchain_core.runnables import RunnableLambda,RunnablePassthrough
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain.schema.output_parser import StrOutputParser

def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

rag_chain = (
    {
        "context": compression_retriever | RunnableLambda(format_docs),
        "query": RunnablePassthrough()
    }
    | prompt
    | chat_model
    | StrOutputParser()
)

In [ ]:
rag_chain.invoke("What are the recommended steps for checking belt tension and ensuring proper motor pulley alignment on both the Original Prusa MK-series and Original Prusa XL printers?")

'For the Original Prusa XL, to check the belt tension, follow the guidelines in the dedicated article. Ensure that the belt clamps are in good visible status and remove the front cover to check for visual damage. Also, check the belt tension using the guidelines in the dedicated article and ensure that the belt clamps are secure.'

### Generate Evaluation dataset and get response time

In [14]:
import pandas as pd
import time
from ragas import EvaluationDataset

def get_eval_ds(path, retriever, rag_chain):

    testset = pd.read_csv(path)
    queries = testset["user_input"].to_list()
    references = testset["reference"].to_list()

    # generate evaluation dataset
    rs_times = []
    dataset = []

    for query, reference in zip(queries, references):

        
        relevant_docs = [doc.page_content for doc in retriever.invoke(query)]
        # measure response time
        t0 = time.perf_counter()
        response = rag_chain.invoke(query)
        t1 = time.perf_counter()
        
        dataset.append(
            {
                "user_input": query,
                "retrieved_contexts": relevant_docs,
                "response": response,
                "reference": reference,
            }
        )
        rs_times.append(t1 -t0)

    eval_ds = EvaluationDataset.from_list(dataset)

    return eval_ds, rs_times

In [ ]:
from ragas import evaluate, RunConfig
from ragas.metrics import LLMContextPrecisionWithReference, LLMContextRecall, AnswerAccuracy, AnswerRelevancy, Faithfulness, FactualCorrectness
from eval_utils import load_eval_model

def get_eval_result(eval_ds, metrics):
    evaluator_llm = load_eval_model()

    run_config = RunConfig(
    timeout=600,      
    max_workers=2,    
    max_retries=2,    
    max_wait=600,     
)
    result = evaluate(
        dataset=eval_ds,
        metrics=metrics,
        llm=evaluator_llm,
        run_config=run_config,
        )
    
    return result

metrics=[LLMContextPrecisionWithReference(),
         LLMContextRecall(), 
         AnswerAccuracy(),
         AnswerRelevancy(),
         Faithfulness(),
         FactualCorrectness(mode = 'f1', atomicity='high', coverage='high')]

In [16]:
data_path = "evaluate_dataset/test_dataset.csv"

eval_ds, rs_times = get_eval_ds(data_path, compression_retriever, rag_chain)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [17]:
result = get_eval_result(eval_ds, metrics)
result

Evaluating: 100%|██████████| 240/240 [29:28<00:00,  7.37s/it]


{'llm_context_precision_with_reference': 0.6934, 'context_recall': 0.6873, 'nv_accuracy': 0.4000, 'answer_relevancy': 0.9104, 'faithfulness': 0.8148, 'factual_correctness(mode=f1)': 0.3452}

In [ ]:
df = result.to_pandas()
df['response_time'] = rs_times
df.to_csv("./evaluate_results/09_prompt_test/final_prompt.csv")

In [ ]:
data_path = "evaluate_dataset/test_dataset_corrected.csv"

eval_ds_cor, rs_times_cor = get_eval_ds(data_path, compression_retriever, rag_chain)
result_cor = get_eval_result(eval_ds, metrics)
print(result_cor)
df_cor = eval_ds_cor.to_pandas()
df_cor['response_time'] = rs_times_cor
df_cor.to_csv("./evaluate_results/09_prompt_test/final_prompt_cor.csv")

Evaluating: 100%|██████████| 240/240 [31:41<00:00,  7.92s/it]


{'llm_context_precision_with_reference': 0.6995, 'context_recall': 0.7027, 'nv_accuracy': 0.3875, 'answer_relevancy': 0.9090, 'faithfulness': 0.7988, 'factual_correctness(mode=f1)': 0.3195}
